# 05 · Tensors & autograd

Pairs with `GUIDE.md` steps 1-4. A tensor is *storage + shape + strides + dtype +
device*, and PyTorch records every op into a graph so `backward()` can fill `.grad`.

In [ ]:
import torch
from gpulab.learn import inspect as I
HAS_CUDA = torch.cuda.is_available()
dev = torch.device("cuda" if HAS_CUDA else "cpu")
print("device:", dev, "| CUDA:", HAS_CUDA)
if not HAS_CUDA:
    print("No CUDA here - cells run on CPU; the timing/memory numbers are only")
    print("meaningful on your RTX 3060. Run this notebook there for the real story.")

## 1. Tensor anatomy: storage + view

In [ ]:
x = torch.arange(12).reshape(3, 4).float()
I.describe_tensor(x, "x")
I.show_storage(x.t(), "x.t()")
print("x and x.t() share storage:", I.shares_memory(x, x.t()))
print("x and x+0  share storage:", I.shares_memory(x, x + 0))

In [ ]:
# Your turn: make a non-contiguous view (x.t()), call .contiguous(), and show the
# data_ptr changed (a copy happened). Use I.describe_tensor before/after.
# TODO

## 2. dtype = memory and speed

In [ ]:
for t in (x.float(), x.half(), x.bfloat16(), x.double()):
    print(f"{str(t.dtype):16s} {t.element_size()} bytes/elem  total {t.numel()*t.element_size()} B")

## 3. Device & the sync trap

In [ ]:
g = x.to(dev)                 # host->device copy if CUDA
y = (g * 2).sum()             # queued (async on CUDA)
val = y.item()                # forces a sync: CPU waits for the GPU
print("sum =", val)
if HAS_CUDA: I.cuda_mem("after move + reduce")

## 4. Autograd: the graph

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
L = a * b + a**2              # dL/da = b + 2a = 7 ; dL/db = a = 2
L.backward()
I.trace_graph(L)
print("a.grad", a.grad.item(), "(expect 7) | b.grad", b.grad.item(), "(expect 2)")

In [ ]:
# Your turn:
# (1) wrap the forward in `with torch.no_grad():` and check L.grad_fn is None.
# (2) call backward() TWICE without zeroing a.grad and watch it accumulate.
# TODO

> **Concepts to note** (copy into your own theory notebook):
> - View vs copy; `.contiguous()` forces a real copy (same as NumPy strides).
> - GPU ops are ASYNC; `.item()`/`.cpu()`/print force a sync (costly in a hot loop).
> - Autograd builds a graph of `grad_fn` nodes; `backward()` walks it.
> - Grads ACCUMULATE -> that's why loops call `zero_grad()` every step.
> - `CrossEntropyLoss` wants integer targets, not one-hot.